Notebook that handles a bunch of non-workforce special demand categories:

- Education (higher ed)
  - Split between existing-residential and campus housing
  - prefix: `UNI`
    - (Following Depot, it covers: universities, technical colleges (polytechnics & ITE) and junior colleges; currently will only implement everything except JCs - that data is hard to come by)
- Education (all other)
  - all existing-residential
- Airports
  - Split between existing-residential and hotels
  - prefix: `AIR`
- Seaports: NIL
- Tourism
  - Split between existing-residential and hotels
  - prefixes:
    - `AMU` for amusement parks; `ZOO`; `PRK` for parks
    - There are a bunch more, see Depot repo

See `docs/non-workforce-special-demand-architecture.md`.

# Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src import data_filepaths
from src.data_manual.airports import AIRPORTS
from src.utils.special_demand import airport_destinations

ALLOW_EXPORT = True


# Education (Tertiary)

This is just be a preliminary run, since the improved building floors and classification data for dorms on NUS/NTU campuses
are not in the current Overture data release.

Future runs will need to include those housing data. For now, it's all additive onto existing residential points.

# Airports/Seaports

Only airports for now, seaport data is quite difficult to handle.

## Airports

Annual passenger totals in `src.data_manual.airports` are modeled demand inputs.
Divide by 365 without adding a return-trip multiplier. Terminal capacities determine
within-airport shares. 

SIN's hotel share is 0.6, approximated from international visitor arrival count data.
XSP's hotel share remains a provisional 0.5. This export contains no workforce demand.

Destination IDs follow `AIR_<terminal-name>`, with underscores between words,
for example `AIR_SIN_Terminal_1` and `AIR_XSP_Terminal`.


In [2]:
HOTEL_SHARES = {"SIN": 0.6, "XSP": 0.5}
DAYS_PER_YEAR = 365

airport_demand = airport_destinations(AIRPORTS, HOTEL_SHARES, days_per_year=DAYS_PER_YEAR)
airport_demand[["name", "daily_demand", "hotel_share"]]


,name,daily_demand,hotel_share
point_id,,,
AIR_SIN_T1,SIN T1,16803.652968,0.6
AIR_SIN_T2,SIN T2,19604.261796,0.6
AIR_SIN_T3,SIN T3,15403.348554,0.6
AIR_SIN_T4,SIN T4,11202.435312,0.6
AIR_XSP,XSP,400.000000,0.5


In [3]:
if ALLOW_EXPORT:
    airport_path = Path(data_filepaths.AIRPORT_DEMAND)
    airport_path.parent.mkdir(parents=True, exist_ok=True)
    airport_demand.to_parquet(airport_path)
    print(f"Exported {len(airport_demand)} terminal demand points to {airport_path}")


Exported 5 terminal demand points to src\data_intermediate\special_demand\nonworkforce\airport_demand.parquet


# Tourism